# GetGTExTables — build the merged GTEx tables in `code/analysis_files/`

Cleaned from `ds_v_dge_scratch.ipynb`, which produced these four tables through
ad-hoc cells that wrote into the notebook's own directory; the files were then
moved to `code/analysis_files/` by hand.

| Output | Consumed by |
|---|---|
| `GTEx.psi.tsv.gz` | **Fig. 2c heatmap** (`Figure2_heatmap_helpers.R`) |
| `GTEx.psi_pvals.tsv.gz` | supplementary / exploratory |
| `GTEx.exp_pvals.tsv.gz` | supplementary / exploratory |
| `GTEx.cluster_counts.tsv.gz` | supplementary / exploratory |

Each table merges 1,225 pairwise tissue comparisons on
`intron, cluster, itype, ctype, gene_name, gene_id`. Where a tissue appears in
more than one comparison, its values are collapsed with the **median**.

---

### ⚠ Source-directory discrepancy carried over from the original

`ds_v_dge_scratch.ipynb` did not read all four tables from the same place:

* `GTEx.psi.tsv.gz` was built from **`code/tmp/ds_v_dge/`**
* the other three were built from **`code/results/ds_v_dge_confounder/tables/`**
  (the output of `rules/ds-dge.smk :: PrepareTablesForHeatmap_generalized`)

Both directories still exist, both hold 11,025 files with identical row and
column counts, **but their contents differ** — the same comparison has a
different md5 in each. They are two different runs, not copies.

`SOURCE_DIR` below therefore defaults to `tmp/ds_v_dge/` so this notebook
reproduces the `GTEx.psi.tsv.gz` that Fig. 2c currently uses. Switch it to
`TABLES_RULE_DIR` to rebuild from the workflow output instead — but expect the
values to change, and re-check Fig. 2c if you do.

In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = '/project/yangili1/cfbuenabadn/leafcutter2_paper'

# The workflow output (rules/ds-dge.smk :: PrepareTablesForHeatmap_generalized)
TABLES_RULE_DIR = f'{BASE}/code/results/ds_v_dge_confounder/tables'
# The directory ds_v_dge_scratch.ipynb actually used for the psi table
TABLES_TMP_DIR = f'{BASE}/code/tmp/ds_v_dge'

# See the note above before changing this.
SOURCE_DIR = TABLES_TMP_DIR

OUT_DIR = f'{BASE}/code/analysis_files'

# Extension -> output filename. psi feeds the Fig. 2c heatmap.
TABLES = {
    'psi': 'GTEx.psi.tsv.gz',
    'psi_p': 'GTEx.psi_pvals.tsv.gz',
    'exp_p': 'GTEx.exp_pvals.tsv.gz',
    'cluster_counts': 'GTEx.cluster_counts.tsv.gz',
}

KEYS = ['intron', 'cluster', 'itype', 'ctype', 'gene_name', 'gene_id']

os.makedirs(OUT_DIR, exist_ok=True)
print('reading from :', SOURCE_DIR)
print('writing to   :', OUT_DIR)

In [ ]:
def list_comparisons(source_dir=SOURCE_DIR):
    """The pairwise tissue comparisons available, from the .psi.* filenames."""
    files_list = os.listdir(source_dir)
    return sorted({x.split('.psi.')[0] for x in files_list if '.psi.' in x})


def merge_dataframes(df1, df2):
    """Outer-merge two comparison tables on the intron keys.

    A tissue can appear in several comparisons. When that happens the merge
    produces `<tissue>_x` and `<tissue>_y`, which are collapsed to their median
    and the suffixed columns dropped.
    """
    df = pd.merge(df1, df2, left_on=KEYS, right_on=KEYS, how='outer')

    if any(x.endswith('_x') for x in df.columns):
        duplicate_tissues = [x.split('_x')[0] for x in df.columns if x.endswith('_x')]
        for tissue in duplicate_tissues:
            df[tissue] = list(np.array(df[[f'{tissue}_x', f'{tissue}_y']].median(axis=1)))

    return df[[x for x in df.columns if not (x.endswith('_x') or x.endswith('_y'))]]


def merge_all_dataframes(ext, comparisons, source_dir=SOURCE_DIR):
    """Merge every comparison's `.{ext}.tsv.gz` into one wide table."""
    df = pd.DataFrame(columns=KEYS)
    for comparison in tqdm(comparisons, position=0, leave=True, desc=ext):
        df_ = pd.read_csv(f'{source_dir}/{comparison}.{ext}.tsv.gz', sep='\t')
        df = merge_dataframes(df, df_)
    return df

In [ ]:
comparisons = list_comparisons()
print(f'{len(comparisons)} pairwise comparisons found')
print('first three:', comparisons[:3])

In [ ]:
# Builds all four tables. Each is a 1,225-way merge, so this is slow and
# memory-hungry; the originals were built one at a time with `del` in between,
# which is what the loop does here.

for ext, out_name in TABLES.items():
    df = merge_all_dataframes(ext, comparisons)
    out_path = f'{OUT_DIR}/{out_name}'
    df.to_csv(out_path, sep='\t', header=True, index=False)
    print(f'{out_name:<28} {df.shape[0]:>8} rows x {df.shape[1]:>3} cols  ->  {out_path}')
    del df